In [0]:
%pip install mlflow databricks-sdk
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Mosaic AI Agent - Databricks

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Connect to Databricks workspace
client = WorkspaceClient()

response = client.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        ChatMessage(
            role=ChatMessageRole.USER,
            content="Say hello in one sentence!"
        )
    ],
    max_tokens=100
)

print(" Mosaic AI LLM connected!")
print(f" Response: {response.choices[0].message.content}")

 Mosaic AI LLM connected!
 Response: Hello, it's nice to meet you and I'm looking forward to our conversation.


In [0]:
# ============================================
# Real Mosaic AI Agent - Text to SQL approach
# ============================================

import uuid
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse
)
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Table schema - so LLM knows what tables and columns exist
SCHEMA = """
You have access to these Bronze Delta tables in Databricks:

1. bronze_transactions
   - transactionID (int)
   - customerID (int)
   - franchiseID (int)
   - dateTime (timestamp)
   - product (string)
   - quantity (int)
   - unitPrice (float)
   - totalPrice (float)
   - paymentMethod (string)

2. bronze_customers
   - customerID (int)
   - first_name (string)
   - last_name (string)
   - email_address (string)
   - city (string)
   - state (string)
   - country (string)
   - continent (string)
   - gender (string)

3. bronze_franchises
   - franchiseID (int)
   - franchiseName (string)
   - city (string)
   - state (string)
   - country (string)
"""

class BronzeDataAgent(ResponsesAgent):
    
    def __init__(self):
        self.client = WorkspaceClient()
        self.model = "databricks-meta-llama-3-3-70b-instruct"
    
    def generate_sql(self, question):
        """Step 1 - LLM generates SQL from natural language question"""
        
        prompt = f"""You are a Spark SQL expert.

Given these tables:
{SCHEMA}

Generate a valid Spark SQL query to answer this question:
{question}

Rules:
- Return ONLY the SQL query, nothing else
- No explanations, no markdown, no backticks
- Use proper Spark SQL syntax
- Always add LIMIT 20 unless question asks for specific count
"""
        
        response = self.client.serving_endpoints.query(
            name=self.model,
            messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
            max_tokens=300
        )
        
        sql = response.choices[0].message.content.strip()
        print(f"Generated SQL: {sql}")
        return sql
    
    def run_sql(self, sql):
        """Step 2 - Run generated SQL on Bronze Delta tables"""
        try:
            df = spark.sql(sql)
            return df.toPandas().to_string()
        except Exception as e:
            return f"SQL Error: {str(e)}"
    
    def explain_result(self, question, data):
        """Step 3 - LLM explains the result in plain English"""
        
        prompt = f"""You are a helpful data analyst for a bakehouse business.

Based on this data:
{data}

Answer this question clearly and concisely:
{question}"""
        
        response = self.client.serving_endpoints.query(
            name=self.model,
            messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
            max_tokens=500
        )
        
        return response.choices[0].message.content
    
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        
        user_question = request.input[-1].content
        
        # Step 1 - Generate SQL from question
        sql = self.generate_sql(user_question)
        
        # Step 2 - Run SQL on Bronze Delta tables
        data = self.run_sql(sql)
        
        # Step 3 - Explain result in plain English
        answer = self.explain_result(user_question, data)
        
        return ResponsesAgentResponse(output=[
            {
                "id": str(uuid.uuid4()),
                "type": "message",
                "role": "assistant",
                "content": [
                    {
                        "type": "output_text",
                        "text": answer,
                        "annotations": []
                    }
                ]
            }
        ])

print("Mosaic AI Agent with Text-to-SQL ready!")

Mosaic AI Agent with Text-to-SQL ready!


In [0]:

# Test Mosaic AI Agent

import mlflow
import time

agent = BronzeDataAgent()

mlflow.set_experiment("/Users/theepankumar72@gmail.com/mosaic_ai_agent")

def ask_mosaic_agent(question):
    with mlflow.start_run():
        print(f"\nQuestion: {question}")     
        start_time = time.time()
        request = ResponsesAgentRequest(
            input=[{"role": "user", "content": question}]
        )
        
        response = agent.predict(request)
        duration = time.time() - start_time
        
        # get answer from content
        answer = response.output[0].content[0]["text"]
        
        mlflow.log_param("question", question)
        mlflow.log_param("model", "databricks-meta-llama-3-3-70b-instruct")
        mlflow.log_metric("response_time_seconds", round(duration, 2))
        mlflow.log_text(str(answer), "answer.txt")
        
        print(f"Answer: {answer}")
        print(f"Response time: {round(duration, 2)}s")
        print(f"MLflow tracking: done")

ask_mosaic_agent("What are the top 3 best selling products by revenue?")
ask_mosaic_agent("Which continent has the most customers?")
ask_mosaic_agent("How many franchises do we have?")


Question: What are the top 3 best selling products by revenue?
Generated SQL: SELECT product, SUM(totalPrice) as total_revenue 
FROM bronze_transactions 
GROUP BY product 
ORDER BY total_revenue DESC 
LIMIT 3
Answer: The top 3 best-selling products by revenue are:

1. Golden Gate Ginger ($11,595)
2. Outback Oatmeal ($11,199)
3. Austin Almond Biscotti ($11,148)
Response time: 12.38s
MLflow tracking: done

Question: Which continent has the most customers?
Generated SQL: SELECT continent, COUNT(customerID) as customer_count FROM bronze_customers GROUP BY continent ORDER BY customer_count DESC LIMIT 20
Answer: The continent with the most customers is Oceania, with 108 customers.
Response time: 2.48s
MLflow tracking: done

Question: How many franchises do we have?
Generated SQL: SELECT COUNT(franchiseID) FROM bronze_franchises LIMIT 20
Answer: We have 48 franchises.
Response time: 4.73s
MLflow tracking: done


[Trace(trace_id=tr-19a8c26c8e923f8009b956814e137f41), Trace(trace_id=tr-40c47a866c0dcd2fcd26b171a2b4eab0), Trace(trace_id=tr-0fdc01eb2307d57bbaba16273b26507a)]